# Assignment 1: Netflix churn dashboard

Run with `uv run python main.py app`. Everything here is loaded from `artifacts/`, written by `train.ipynb`.
Nothing is trained or refit: the three models are rebuilt from their saved weights and scored on the saved test set.

Three tabs:

1. **Model comparison**: predicted probability against the actual outcome for all three implementations, how far
   they agree with each other, and their weights side by side.
2. **Distributions**: any raw or engineered column, split by churned / stayed, on the training set.
3. **Threshold & cost**: move the decision threshold and the cost assumptions, and watch the confusion matrix and
   the retention cost per 1,000 subscribers change.

## 1. Loading the artifacts

In [ ]:
import json
from pathlib import Path

import gradio as gr
import joblib
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import torch
import torch.nn as nn
from sklearn.metrics import log_loss, roc_auc_score

ARTIFACTS = Path("artifacts")
meta = json.loads((ARTIFACTS / "meta.json").read_text())
FEATURES, TARGET, COSTS, THRESHOLD = meta["features"], meta["target"], meta["costs"], meta["threshold"]

scaler = joblib.load(ARTIFACTS / "scaler.joblib")
sk_model = joblib.load(ARTIFACTS / "sklearn_model.joblib")
manual = torch.load(ARTIFACTS / "manual_torch.pt")
standard_model = nn.Linear(len(FEATURES), 1)
standard_model.load_state_dict(torch.load(ARTIFACTS / "standard_torch.pt"))
standard_model.eval()

train_data = pd.read_csv(ARTIFACTS / "train_data.csv")
test_data = pd.read_csv(ARTIFACTS / "test_data.csv")
train_data["outcome"] = np.where(train_data[TARGET] == 1, "churned", "stayed")
print("features:", FEATURES)
print("train:", train_data.shape, "| test:", test_data.shape, "| shipped threshold:", THRESHOLD)

## 2. Scoring the test set with the saved models

In [ ]:
X_test = scaler.transform(test_data[FEATURES]).astype(np.float32)
Xt_test = torch.tensor(X_test)
y_test = test_data[TARGET].to_numpy()

with torch.no_grad():
    PROBS = {
        "scikit-learn": sk_model.predict_proba(X_test)[:, 1],
        "manual torch": torch.sigmoid(Xt_test @ manual["w"] + manual["b"]).numpy(),
        "standard torch": torch.sigmoid(standard_model(Xt_test).squeeze(-1)).numpy(),
    }
MODELS = list(PROBS)

WEIGHTS = pd.DataFrame(
    {
        "scikit-learn": sk_model.coef_[0],
        "manual torch": manual["w"].numpy(),
        "standard torch": standard_model.weight.detach().numpy()[0],
    },
    index=FEATURES,
)

## 3. Cost helpers

In [ ]:
def cost_table(offer_cost, churn_cost, save_rate):
    return {"TP": offer_cost + (1 - save_rate) * churn_cost, "FP": offer_cost, "FN": churn_cost, "TN": 0.0}


def confusion(y, p, t):
    offer = p >= t
    tp = int(np.sum(offer & (y == 1)))
    fp = int(np.sum(offer & (y == 0)))
    fn = int(np.sum(~offer & (y == 1)))
    tn = int(np.sum(~offer & (y == 0)))
    return tp, fp, fn, tn


def cost_per_1000(y, p, t, costs):
    tp, fp, fn, tn = confusion(y, p, t)
    return 1000 * (tp * costs["TP"] + fp * costs["FP"] + fn * costs["FN"] + tn * costs["TN"]) / len(y)


def optimal_threshold(costs):
    gain_fp = costs["FP"] - costs["TN"]
    gain_fn = costs["FN"] - costs["TP"]
    return gain_fp / (gain_fp + gain_fn) if gain_fp + gain_fn > 0 else 1.0


DEFAULT_OFFER = COSTS["FP"]
DEFAULT_CHURN = COSTS["FN"]
DEFAULT_SAVE = 1 - (COSTS["TP"] - COSTS["FP"]) / COSTS["FN"]

## 4. Model comparison views

In [ ]:
def metrics_table():
    rows = []
    baseline = {"naive baseline": np.full(len(y_test), train_data[TARGET].mean())}
    for name, p in (baseline | PROBS).items():
        tp, fp, fn, tn = confusion(y_test, p, THRESHOLD)
        rows.append(
            {
                "model": name,
                "AUC": round(roc_auc_score(y_test, p), 4),
                "log loss": round(log_loss(y_test, p), 4),
                f"recall @ {THRESHOLD}": round(tp / (tp + fn), 3),
                f"precision @ {THRESHOLD}": round(tp / (tp + fp), 3),
                "EUR / 1,000": round(cost_per_1000(y_test, p, THRESHOLD, COSTS)),
            }
        )
    return pd.DataFrame(rows)


def prediction_vs_actual():
    long = pd.concat(
        pd.DataFrame({"model": name, "predicted churn probability": p, "actual": np.where(y_test == 1, "churned", "stayed")})
        for name, p in PROBS.items()
    )
    fig = px.strip(
        long, x="model", y="predicted churn probability", color="actual",
        color_discrete_map={"churned": "#d62728", "stayed": "#1f77b4"},
        title="Predicted probability vs. actual outcome (test set)",
    )
    fig.add_hline(y=THRESHOLD, line_dash="dash", annotation_text=f"shipped threshold {THRESHOLD}")
    return fig


def agreement_plot():
    fig = go.Figure()
    for name in MODELS[1:]:
        fig.add_trace(go.Scatter(x=PROBS["scikit-learn"], y=PROBS[name], mode="markers", name=name, opacity=0.5))
    fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", name="perfect agreement", line=dict(dash="dash", color="black")))
    gap = max(np.abs(PROBS["scikit-learn"] - PROBS[n]).max() for n in MODELS[1:])
    fig.update_layout(
        title=f"PyTorch vs. scikit-learn probabilities (largest gap {gap:.5f})",
        xaxis_title="scikit-learn probability", yaxis_title="PyTorch probability",
    )
    return fig


def weights_plot():
    long = WEIGHTS.reset_index(names="feature").melt(id_vars="feature", var_name="model", value_name="weight")
    return px.bar(long, x="weight", y="feature", color="model", barmode="group", orientation="h",
                  title="Learned weights (standardised features)")

## 5. Distribution views

In [ ]:
DIST_COLUMNS = [
    "watch_hours", "avg_watch_time_per_day", "last_login_days", "number_of_profiles", "age",
    "subscription_type", "payment_method", "device", "region", "gender", "favorite_genre",
] + FEATURES


def distribution_plot(col):
    fig = px.histogram(
        train_data, x=col, color="outcome", barmode="overlay" if pd.api.types.is_numeric_dtype(train_data[col]) else "group",
        opacity=0.7, nbins=60, color_discrete_map={"churned": "#d62728", "stayed": "#1f77b4"},
        title=f"{col} by outcome (training set)",
    )
    return fig


def target_plot():
    counts = train_data["outcome"].value_counts().reset_index()
    return px.bar(counts, x="outcome", y="count", color="outcome",
                  color_discrete_map={"churned": "#d62728", "stayed": "#1f77b4"},
                  title=f"Target balance (training set, churn rate {train_data[TARGET].mean():.1%})")

## 6. Threshold and cost view

In [ ]:
def threshold_view(model, t, offer_cost, churn_cost, save_rate):
    costs = cost_table(offer_cost, churn_cost, save_rate)
    p = PROBS[model]
    tp, fp, fn, tn = confusion(y_test, p, t)

    cm = px.imshow(
        [[tn, fp], [fn, tp]], x=["no offer", "offer"], y=["stayed", "churned"],
        text_auto=True, color_continuous_scale="Blues", labels=dict(x="decision", y="actual", color="subscribers"),
        title=f"Confusion matrix, {model} @ {t:.2f}",
    )

    t_star = optimal_threshold(costs)
    grid = np.linspace(0, 1, 101)
    curve = go.Figure(go.Scatter(x=grid, y=[cost_per_1000(y_test, p, g, costs) for g in grid], name="cost"))
    curve.add_vline(x=t, line_color="red", annotation_text=f"current {t:.2f}")
    curve.add_vline(x=t_star, line_dash="dash", annotation_text=f"t* {t_star:.2f}", annotation_position="bottom right")
    curve.update_layout(title="Retention cost vs. threshold (test set)", xaxis_title="threshold", yaxis_title="EUR per 1,000")

    nobody = cost_per_1000(y_test, p, np.inf, costs)
    everyone = cost_per_1000(y_test, p, -np.inf, costs)
    current = cost_per_1000(y_test, p, t, costs)
    precision = f"{tp / (tp + fp):.1%}" if tp + fp else "n/a"
    summary = f"""
### EUR {current:,.0f} per 1,000 subscribers

| | |
|---|---|
| offers sent | {tp + fp} of {len(y_test)} ({(tp + fp) / len(y_test):.1%}) |
| churners caught (recall) | {tp / (tp + fn):.1%} |
| offers that reached a churner (precision) | {precision} |
| missed churners | {fn} |
| saving vs. offer to nobody | EUR {nobody - current:,.0f} |
| saving vs. offer to everyone | EUR {everyone - current:,.0f} |
| cost-optimal threshold t* for these costs | {t_star:.3f} |
"""
    return cm, curve, summary

## 7. The app

In [ ]:
with gr.Blocks(title="Netflix churn: logistic regression three ways") as demo:
    gr.Markdown(
        "# Netflix churn: one logistic regression, three implementations\n"
        "Retention campaign: subscribers flagged as likely to churn get a one-month-free offer. "
        "All numbers below are on the held-out test set, scored by the saved models."
    )

    with gr.Tab("Model comparison"):
        gr.Dataframe(value=metrics_table(), label="Test set metrics")
        gr.Plot(value=prediction_vs_actual())
        with gr.Row():
            gr.Plot(value=agreement_plot())
            gr.Plot(value=weights_plot())

    with gr.Tab("Distributions"):
        column = gr.Dropdown(DIST_COLUMNS, value="avg_watch_time_per_day", label="Column")
        dist = gr.Plot(value=distribution_plot("avg_watch_time_per_day"))
        gr.Plot(value=target_plot())
        column.change(distribution_plot, column, dist)

    with gr.Tab("Threshold & cost"):
        with gr.Row():
            model = gr.Dropdown(MODELS, value="scikit-learn", label="Model")
            threshold = gr.Slider(0, 1, value=THRESHOLD, step=0.01, label="Decision threshold")
        with gr.Row():
            offer_cost = gr.Number(value=round(DEFAULT_OFFER, 2), label="Offer cost, EUR (one free month)")
            churn_cost = gr.Number(value=round(DEFAULT_CHURN, 2), label="Lost subscriber, EUR (6 months of fees)")
            save_rate = gr.Slider(0, 1, value=round(DEFAULT_SAVE, 2), step=0.05, label="Share of offered churners who stay")
        cm0, curve0, summary0 = threshold_view("scikit-learn", THRESHOLD, DEFAULT_OFFER, DEFAULT_CHURN, DEFAULT_SAVE)
        with gr.Row():
            cm_plot = gr.Plot(value=cm0)
            summary = gr.Markdown(value=summary0)
        curve_plot = gr.Plot(value=curve0)

        inputs = [model, threshold, offer_cost, churn_cost, save_rate]
        for component in inputs:
            component.change(threshold_view, inputs, [cm_plot, curve_plot, summary])

demo.launch()